# Assignment 5: Predict Stock Market Trend using Artificial Neural Network
Sliding window: last N days' prices → next day's price. Trend = UP if predicted > today.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

In [ ]:
def load_prices(choice):
    if choice == '2':
        path = input("Enter CSV file path (must have a 'Close' column): ")
        return pd.read_csv(path)['Close'].dropna().values.astype(float)
    np.random.seed(42)                               # synthetic stock-like prices
    return 1000 + np.cumsum(np.random.randn(500) * 10)

def make_windows(scaled, window):
    X, y = [], []
    for i in range(window, len(scaled)):
        X.append(scaled[i - window:i])               # previous 'window' days
        y.append(scaled[i])                          # next day
    return np.array(X), np.array(y)

def train_model(prices, window):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(prices.reshape(-1, 1)).flatten()
    X, y = make_windows(scaled, window)
    split = int(0.8 * len(X))                        # time-order split, no shuffle
    model = MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu',
                         solver='adam', max_iter=1000, random_state=42)
    model.fit(X[:split], y[:split])
    return model, scaler, scaled, X[split:], y[split:]

In [ ]:
prices, window = None, 5
model = None

while True:
    print("\n========= STOCK PREDICTION (ANN) MENU =========")
    print("1. Load data (synthetic / CSV)")
    print("2. Set window size")
    print("3. Train ANN model")
    print("4. Evaluate on test data (RMSE, trend accuracy)")
    print("5. Predict next day trend")
    print("6. Plot actual vs predicted")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            src = input("1 = Synthetic data, 2 = CSV file: ").strip()
            prices = load_prices(src)
            model = None
            print(f"Loaded {len(prices)} prices. Last price = {prices[-1]:.2f}")
        elif ch == '2':
            window = int(input("Enter window size (e.g. 5): "))
            model = None
        elif ch == '3':
            if prices is None: print("Load data first (option 1)."); continue
            model, scaler, scaled, X_test, y_test = train_model(prices, window)
            print("Model trained. Hidden layers = (32, 16), activation = ReLU, optimizer = Adam")
        elif ch in ('4', '5', '6'):
            if model is None: print("Train the model first (option 3)."); continue
            pred   = scaler.inverse_transform(model.predict(X_test).reshape(-1, 1)).flatten()
            actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
            today  = scaler.inverse_transform(X_test[:, -1].reshape(-1, 1)).flatten()
            if ch == '4':
                rmse = np.sqrt(mean_squared_error(actual, pred))
                trend_acc = np.mean((pred > today) == (actual > today)) * 100
                print(f"RMSE           : {rmse:.2f}")
                print(f"Trend accuracy : {trend_acc:.2f} %")
            elif ch == '5':
                nxt = model.predict(scaled[-window:].reshape(1, -1)).reshape(-1, 1)
                nxt = scaler.inverse_transform(nxt)[0, 0]
                print(f"Last price: {prices[-1]:.2f}, Predicted next: {nxt:.2f} -> "
                      f"Trend: {'UP' if nxt > prices[-1] else 'DOWN'}")
            else:
                plt.plot(actual, label='Actual'); plt.plot(pred, label='Predicted')
                plt.title('Stock Price Prediction using ANN'); plt.legend(); plt.show()
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except (ValueError, FileNotFoundError, KeyError) as e:
        print("Error:", e)